In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix
import joblib
import sys
sys.path.append('../src')

# Load data
train = pd.read_csv('../data/processed/train_processed.csv')

# Rebuild features for analysis
tfidf = TfidfVectorizer(max_features=300, ngram_range=(1,2), min_df=2, sublinear_tf=True)
X_text = tfidf.fit_transform(train['cleaned_text'])

METADATA_COLS = ['sleep_hours', 'energy_level', 'stress_level', 'duration_min',
                 'time_of_day_enc', 'reflection_quality_enc', 'previous_day_mood_enc']
ambience_cols = [c for c in train.columns if c.startswith('ambience_type_')]
face_cols = [c for c in train.columns if c.startswith('face_emotion_hint_')]
all_meta = METADATA_COLS + ambience_cols + face_cols

X_meta = csr_matrix(train[all_meta].values.astype(float))
X_combined = hstack([X_text, X_meta])

# Train model and get predictions on training set
le = LabelEncoder()
y = le.fit_transform(train['emotional_state'])

model = XGBClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.05,
    subsample=0.7, colsample_bytree=0.7,
    random_state=42, eval_metric='mlogloss', verbosity=0
)
model.fit(X_combined, y)
y_pred = le.inverse_transform(model.predict(X_combined))
proba = model.predict_proba(X_combined)
confidence = proba.max(axis=1)

# Add predictions back to dataframe
train['predicted_state'] = y_pred
train['confidence'] = confidence
train['correct'] = train['predicted_state'] == train['emotional_state']

print(f"Overall accuracy on train: {train['correct'].mean():.4f}")
print(f"Total errors: {(~train['correct']).sum()}")

Overall accuracy on train: 0.8767
Total errors: 148


In [2]:
# Get all misclassified rows
errors = train[~train['correct']].copy()

# Sort by confidence — high confidence wrong predictions are most interesting
errors = errors.sort_values('confidence', ascending=False)

# Show key columns only
cols = ['journal_text', 'emotional_state', 'predicted_state', 
        'confidence', 'stress_level', 'energy_level', 
        'sleep_hours', 'time_of_day', 'reflection_quality']

print("Top 15 High-Confidence Errors (model was sure but wrong):")
print(errors[cols].head(15).to_string())

Top 15 High-Confidence Errors (model was sure but wrong):
                                                                journal_text emotional_state predicted_state  confidence  stress_level  energy_level  sleep_hours time_of_day reflection_quality
870                                                         more clear today           mixed            calm    0.511129             3             3          8.0   afternoon              vague
803                                                   By the end felt heavy.         neutral     overwhelmed    0.486140             1             2          7.0   afternoon              vague
742                                                    helped me plan my day     overwhelmed         focused    0.439051             1             4          6.0     evening              vague
764  Honestly helped me plan my day. Later it changed breathing slowed down.        restless            calm    0.431920             1             5          6.0       ni

In [3]:
# Analyze error patterns
print("Errors by true emotional state:")
print(errors['emotional_state'].value_counts())

print("\nErrors by reflection quality:")
print(errors['reflection_quality'].value_counts())

print("\nErrors by time of day:")
print(errors['time_of_day'].value_counts())

print("\nWord count of misclassified texts:")
errors['word_count'] = errors['journal_text'].str.split().str.len()
print(errors['word_count'].describe())

print("\nMost common misclassified texts:")
print(errors['journal_text'].value_counts().head(10))

Errors by true emotional state:
emotional_state
mixed          31
calm           29
neutral        28
overwhelmed    28
restless       17
focused        15
Name: count, dtype: int64

Errors by reflection quality:
reflection_quality
conflicted    57
vague         52
clear         39
Name: count, dtype: int64

Errors by time of day:
time_of_day
morning          45
night            41
afternoon        37
evening          24
early_morning     1
Name: count, dtype: int64

Word count of misclassified texts:
count    148.000000
mean       5.060811
std        2.680812
min        2.000000
25%        3.000000
50%        4.500000
75%        6.000000
max       15.000000
Name: word_count, dtype: float64

Most common misclassified texts:
journal_text
it was fine                       6
a little lighter                  5
honestly not much change          5
okay session                      5
kinda calm now                    4
not sure what changed             4
back to normal after              3
A